In [1]:
import pandas as pd
import locale
import datetime as dt
import numpy as np

In [2]:
locale.setlocale(locale.LC_TIME, 'fr_FR.UTF-8') 

'fr_FR.UTF-8'

In [3]:
lignes_file = 'Lignes du metro.xlsx'

sheets = ['Ligne 1', 'Ligne 2',  'Ligne 2 Sud','Ligne 3', 'Ligne 4', 'Ligne 5', 'Ligne 6', 'Ligne 7', 'Ligne 8', 
          'Ligne 9', 'Ligne 10', 'Ligne 11', 'Ligne 12', 'Ligne 13', 'Ligne 14', 'Ligne 3bis', 'Ligne 7bis', 
          'Ligne 14 (ancienne)', 'Couloirs', 'Voie des Fêtes et voie navette']

station_extract_file  = 'station_extract.csv'

In [4]:
lignes = []
for sheet in sheets: 
    df = pd.read_excel(lignes_file, sheet_name=sheet)
    df['Ligne'] = sheet
    lignes.append(df)
lignes = pd.concat(lignes)

lignes['Ouverture'] = lignes['Ouverture'].map(lambda x: dt.datetime.strptime(x, '%d %B %Y'))
lignes['Fermeture'] = lignes['Fermeture'].map(lambda x: dt.datetime.strptime(x, '%d %B %Y') if type(x)==str else x)

In [41]:
station_extract = pd.read_csv(station_extract_file, index_col=0)
station_extract.index = [station.replace(' (métro de Paris)', '') for station in station_extract.index]

In [10]:
stations_lignes = sorted(set(lignes['Vers'])|set(lignes['De']))

In [42]:
station_info = {}
for station in stations_lignes: 
    station_info[station] = {}
    
    # coordinates 
    if station in station_extract.index: 
        station_info[station]['latitude'] = station_extract.loc[station, 'latitude']
        station_info[station]['longitude'] = station_extract.loc[station, 'longitude']
    
    # liaisions 
    liaisons = lignes.loc[(lignes['De'] == station)|(lignes['Vers'] == station)]
                                                   
    # ouverture 
    station_info[station]['Ouverture'] = liaisons['Ouverture'].min()
    # fermeture 
    if len(liaisons) == len(liaisons.dropna(subset=['Fermeture'])): 
        station_info[station]['Fermetrue'] = liaisons['Fermeture'].max()

In [49]:
pd.DataFrame(station_info).T.to_csv('station_info_raw.csv')